# 10. Final Pipeline

Финальный пайплайн. Все гиперпараметры тюним через Optuna.

In [ ]:
# ─── Конфиг ───────────────────────────────────────────────────────────
FORCE_RETUNE    = False   # True → перезапустить Optuna даже если кэш есть

THR_IC50        = 500
THR_CC50        = 1500
THR_SI          = 200

N_TRIALS_REG    = 120
N_TRIALS_CLF    =  60

ARTIFACT_PREFIX = 'final_'
SEED            = 42
N_FOLDS         = 5

TARGET_COLS     = ['IC50', 'CC50', 'SI']
RAW_TARGET_COLS = ['IC50, mM', 'CC50, mM', 'SI']
INDEX_COL       = 'index'

# ─── Пути ─────────────────────────────────────────────────────────────
from pathlib import Path
_NB_DIR         = Path().resolve()
REPO_ROOT       = _NB_DIR.parent if _NB_DIR.name == 'notebooks' else _NB_DIR
RAW_DIR         = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR   = REPO_ROOT / 'data' / 'processed'
SUBMISSIONS_DIR = REPO_ROOT / 'data' / 'submissions'
ARTIFACTS_DIR   = REPO_ROOT / 'artifacts'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import warnings
import json
from collections import defaultdict

import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
import xgboost as xgb
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Утилиты

In [ ]:
def log1p_target(y):
    return np.log1p(y)

def expm1_pred(y_log):
    return np.expm1(y_log)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true, float) - np.asarray(y_pred, float)) ** 2)))

def competition_score(y_true_ic50, y_pred_ic50, y_true_cc50, y_pred_cc50, y_true_si, y_pred_si):
    r_ic50 = rmse(y_true_ic50, y_pred_ic50)
    r_cc50 = rmse(y_true_cc50, y_pred_cc50)
    r_si   = rmse(y_true_si,   y_pred_si)
    return {'rmse_ic50': r_ic50, 'rmse_cc50': r_cc50, 'rmse_si': r_si,
            'score': (r_ic50 + r_cc50 + r_si) / 3.0}

## Очистка данных

In [ ]:
_ROUND = 8

def _rename_targets(df):
    return df.rename(columns=dict(zip(RAW_TARGET_COLS, TARGET_COLS)))

def _feature_columns(df):
    drop = {INDEX_COL} | set(TARGET_COLS)
    return [c for c in df.columns if c not in drop]

def _deduplicate(df, feature_cols):
    keys = [tuple(row.round(_ROUND)) for row in df[feature_cols].values]
    groups = defaultdict(list)
    for pos, key in enumerate(keys):
        groups[key].append(pos)
    rows = []
    for positions in groups.values():
        group = df.iloc[positions]
        row = group.iloc[0][feature_cols + [INDEX_COL]].to_dict()
        for tc in TARGET_COLS:
            row[tc] = float(group[tc].median())
        rows.append(row)
    return pd.DataFrame(rows, columns=df.columns.tolist()).reset_index(drop=True)

def clean(prefix='', verbose=True):
    train = _rename_targets(pd.read_csv(RAW_DIR / 'train.csv'))
    test  = _rename_targets(pd.read_csv(RAW_DIR / 'test.csv'))

    feature_cols = _feature_columns(train)
    stds     = train[feature_cols].std(numeric_only=True)
    zero_var = stds.index[stds == 0].tolist()
    train = train.drop(columns=zero_var, errors='ignore')
    test  = test.drop(columns=zero_var, errors='ignore')
    feature_cols = _feature_columns(train)

    medians = train[feature_cols].median(numeric_only=True)
    train[feature_cols] = train[feature_cols].fillna(medians)
    test[feature_cols]  = test[feature_cols].fillna(medians)

    n_raw   = len(train)
    train   = _deduplicate(train, feature_cols)
    n_dedup = len(train)

    train_path = PROCESSED_DIR / f'{prefix}train_clean.csv'
    test_path  = PROCESSED_DIR / f'{prefix}test_clean.csv'
    train.to_csv(train_path, index=False)
    test.to_csv(test_path,   index=False)

    if verbose:
        print(f'[clean] train: {n_raw} → {n_dedup} строк (−{n_raw - n_dedup} дублей)')
        print(f'[clean] test : {len(test)} строк')
        print(f'[clean] фичей: {len(feature_cols) + len(zero_var)} → {len(feature_cols)}')
    return train_path, test_path

## Фичи

In [ ]:
RATIO_PAIRS = [
    ('MolWt', 'HeavyAtomMolWt'),
    ('NumValenceElectrons', 'MolWt'),
    ('BertzCT', 'MolWt'),
]
N_PCA   = 20
_scaler = None
_pca    = None

def build_features(df, feature_cols, fit=False):
    global _scaler, _pca
    X_base = df[feature_cols].values
    if fit:
        _scaler  = StandardScaler()
        X_scaled = _scaler.fit_transform(X_base)
        _pca     = PCA(n_components=N_PCA, random_state=SEED)
        X_pca    = _pca.fit_transform(X_scaled)
    else:
        X_scaled = _scaler.transform(X_base)
        X_pca    = _pca.transform(X_scaled)
    ratios = [
        (df[a] / np.maximum(df[b], 1e-9)).values.reshape(-1, 1)
        for a, b in RATIO_PAIRS
        if a in df.columns and b in df.columns
    ]
    return np.hstack([X_base, X_pca] + ratios)

## Данные

In [4]:
train_path, test_path = clean(verbose=True, prefix=ARTIFACT_PREFIX)

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)
feature_cols = [c for c in train.columns if c not in TARGET_COLS + [INDEX_COL]]

X_train = build_features(train, feature_cols, fit=True)
X_test  = build_features(test,  feature_cols, fit=False)
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

y_ic50 = train['IC50'].values
y_cc50 = train['CC50'].values
y_si   = train['SI'].values

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

[clean] train: 751 → 630 строк (−121 дублей)
[clean] test : 250 строк
[clean] фичей: 210 → 192
[clean] NaN:   train=24, test=12
[clean] → /Users/rg/learning/yandex/s2/ml/chemai-predict-the-cure/data/processed/final_train_clean.csv
[clean] → /Users/rg/learning/yandex/s2/ml/chemai-predict-the-cure/data/processed/final_test_clean.csv
X_train: (630, 215) | X_test: (250, 215)


## Хелперы

In [5]:
# ── LightGBM OOF ──────────────────────────────────────────────────────

def lgb_oof(X, y_raw, params, n_rounds=3000, early=100):
    y_log    = log1p_target(y_raw)
    oof_log  = np.zeros(len(X))
    test_log = np.zeros(len(X_test))
    for tr_idx, va_idx in kf.split(X):
        dtr = lgb.Dataset(X[tr_idx], y_log[tr_idx])
        dva = lgb.Dataset(X[va_idx], y_log[va_idx], reference=dtr)
        m = lgb.train({**params, 'seed': SEED, 'verbose': -1}, dtr, n_rounds,
                      valid_sets=[dva],
                      callbacks=[lgb.early_stopping(early), lgb.log_evaluation(0)])
        oof_log[va_idx] = m.predict(X[va_idx], num_iteration=m.best_iteration)
        test_log += m.predict(X_test, num_iteration=m.best_iteration) / N_FOLDS
    return np.clip(expm1_pred(oof_log), 0, None), np.clip(expm1_pred(test_log), 0, None)


def lgb_clf_oof(X, y_label, params, n_rounds=500, early=50):
    oof_prob  = np.zeros(len(X))
    test_prob = np.zeros(len(X_test))
    for tr_idx, va_idx in kf.split(X):
        dtr = lgb.Dataset(X[tr_idx], y_label[tr_idx])
        dva = lgb.Dataset(X[va_idx], y_label[va_idx], reference=dtr)
        m = lgb.train({**params, 'seed': SEED, 'verbose': -1}, dtr, n_rounds,
                      valid_sets=[dva],
                      callbacks=[lgb.early_stopping(early), lgb.log_evaluation(0)])
        oof_prob[va_idx] = m.predict(X[va_idx])
        test_prob += m.predict(X_test) / N_FOLDS
    return oof_prob, test_prob

In [6]:
# ── XGBoost OOF ───────────────────────────────────────────────────────

def xgb_oof(X, y_raw, params, n_rounds=3000, early=100):
    y_log    = log1p_target(y_raw)
    oof_log  = np.zeros(len(X))
    test_log = np.zeros(len(X_test))
    for tr_idx, va_idx in kf.split(X):
        dtrain = xgb.DMatrix(X[tr_idx], y_log[tr_idx])
        dval   = xgb.DMatrix(X[va_idx], y_log[va_idx])
        m = xgb.train({**params, 'seed': SEED, 'verbosity': 0}, dtrain, n_rounds,
                      evals=[(dval, 'val')], early_stopping_rounds=early,
                      verbose_eval=False)
        oof_log[va_idx] = m.predict(dval)
        test_log += m.predict(xgb.DMatrix(X_test)) / N_FOLDS
    return np.clip(expm1_pred(oof_log), 0, None), np.clip(expm1_pred(test_log), 0, None)


def xgb_oof_weighted(X, y_raw, params, weights, n_rounds=3000, early=100):
    y_log    = log1p_target(y_raw)
    oof_log  = np.zeros(len(X))
    test_log = np.zeros(len(X_test))
    for tr_idx, va_idx in kf.split(X):
        dtrain = xgb.DMatrix(X[tr_idx], y_log[tr_idx], weight=weights[tr_idx])
        dval   = xgb.DMatrix(X[va_idx], y_log[va_idx])
        m = xgb.train({**params, 'seed': SEED, 'verbosity': 0}, dtrain, n_rounds,
                      evals=[(dval, 'val')], early_stopping_rounds=early,
                      verbose_eval=False)
        oof_log[va_idx] = m.predict(dval)
        test_log += m.predict(xgb.DMatrix(X_test)) / N_FOLDS
    return np.clip(expm1_pred(oof_log), 0, None), np.clip(expm1_pred(test_log), 0, None)


def xgb_clf_oof(X, y_label, params, n_rounds=500, early=50):
    oof_prob  = np.zeros(len(X))
    test_prob = np.zeros(len(X_test))
    for tr_idx, va_idx in kf.split(X):
        dtrain = xgb.DMatrix(X[tr_idx], y_label[tr_idx])
        dval   = xgb.DMatrix(X[va_idx], y_label[va_idx])
        m = xgb.train({**params, 'seed': SEED, 'verbosity': 0,
                       'objective': 'binary:logistic', 'eval_metric': 'auc'},
                      dtrain, n_rounds,
                      evals=[(dval, 'val')], early_stopping_rounds=early,
                      verbose_eval=False)
        oof_prob[va_idx] = m.predict(dval)
        test_prob += m.predict(xgb.DMatrix(X_test)) / N_FOLDS
    return oof_prob, test_prob

In [7]:
# ── Optuna: тюнинг ────────────────────────────────────────────────────

def tune_xgb_reg(X, y_raw, n_trials=N_TRIALS_REG, label=''):
    y_log = log1p_target(y_raw)
    def objective(trial):
        p = {
            'objective': 'reg:squarederror', 'eval_metric': 'rmse',
            'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
            'max_depth':         trial.suggest_int('max_depth', 3, 9),
            'min_child_weight':  trial.suggest_int('min_child_weight', 1, 50),
            'subsample':         trial.suggest_float('subsample', 0.4, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.3, 1.0),
            'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.3, 1.0),
            'reg_lambda':        trial.suggest_float('reg_lambda', 0.01, 20.0, log=True),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
            'gamma':             trial.suggest_float('gamma', 0, 5),
            'seed': SEED, 'verbosity': 0,
        }
        scores = []
        for tr_idx, va_idx in kf.split(X):
            dtrain = xgb.DMatrix(X[tr_idx], y_log[tr_idx])
            dval   = xgb.DMatrix(X[va_idx], y_log[va_idx])
            m = xgb.train(p, dtrain, 3000,
                          evals=[(dval, 'val')], early_stopping_rounds=100,
                          verbose_eval=False)
            pred = np.clip(expm1_pred(m.predict(dval)), 0, None)
            scores.append(rmse(y_raw[va_idx], pred))
        return np.mean(scores)
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f'[{label}] best OOF RMSE: {study.best_value:.4f}')
    return {'objective': 'reg:squarederror', 'eval_metric': 'rmse', **study.best_params}


def tune_xgb_clf(X, y_label, n_trials=N_TRIALS_CLF, label=''):
    def objective(trial):
        p = {
            'objective': 'binary:logistic', 'eval_metric': 'auc',
            'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
            'max_depth':        trial.suggest_int('max_depth', 2, 7),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 30),
            'subsample':        trial.suggest_float('subsample', 0.4, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
            'reg_lambda':       trial.suggest_float('reg_lambda', 0.01, 20.0, log=True),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 20.0),
            'seed': SEED, 'verbosity': 0,
        }
        oof_prob = np.zeros(len(X))
        for tr_idx, va_idx in kf.split(X):
            dtrain = xgb.DMatrix(X[tr_idx], y_label[tr_idx])
            dval   = xgb.DMatrix(X[va_idx], y_label[va_idx])
            m = xgb.train(p, dtrain, 500,
                          evals=[(dval, 'val')], early_stopping_rounds=50,
                          verbose_eval=False)
            oof_prob[va_idx] = m.predict(dval)
        return -roc_auc_score(y_label, oof_prob)
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f'[{label} clf] best ROC-AUC: {-study.best_value:.4f}')
    return {'objective': 'binary:logistic', 'eval_metric': 'auc', **study.best_params}


def tune_lgb_reg(X, y_raw, n_trials=N_TRIALS_REG, label=''):
    y_log = log1p_target(y_raw)
    def objective(trial):
        p = {
            'objective': 'regression', 'metric': 'rmse', 'verbose': -1,
            'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
            'num_leaves':       trial.suggest_int('num_leaves', 20, 200),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 100),
            'max_depth':        trial.suggest_int('max_depth', 3, 9),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.3, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.3, 1.0),
            'bagging_freq':     trial.suggest_int('bagging_freq', 1, 10),
            'lambda_l1':        trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
            'lambda_l2':        trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
        }
        scores = []
        for tr_idx, va_idx in kf.split(X):
            dtr = lgb.Dataset(X[tr_idx], y_log[tr_idx])
            dva = lgb.Dataset(X[va_idx], y_log[va_idx], reference=dtr)
            m = lgb.train(p, dtr, 3000, valid_sets=[dva],
                          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            pred = np.clip(expm1_pred(
                m.predict(X[va_idx], num_iteration=m.best_iteration)), 0, None)
            scores.append(rmse(y_raw[va_idx], pred))
        return np.mean(scores)
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f'[{label}] best OOF RMSE: {study.best_value:.4f}')
    return {'objective': 'regression', 'metric': 'rmse', **study.best_params}


def tune_lgb_clf(X, y_label, n_trials=N_TRIALS_CLF, label=''):
    def objective(trial):
        p = {
            'objective': 'binary', 'metric': 'binary_logloss',
            'is_unbalance': True, 'verbose': -1,
            'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
            'num_leaves':       trial.suggest_int('num_leaves', 20, 150),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 50),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.3, 1.0),
            'lambda_l1':        trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
            'lambda_l2':        trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
        }
        oof_prob = np.zeros(len(X))
        for tr_idx, va_idx in kf.split(X):
            dtr = lgb.Dataset(X[tr_idx], y_label[tr_idx])
            dva = lgb.Dataset(X[va_idx], y_label[va_idx], reference=dtr)
            m = lgb.train(p, dtr, 500, valid_sets=[dva],
                          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
            oof_prob[va_idx] = m.predict(X[va_idx])
        return -roc_auc_score(y_label, oof_prob)
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f'[{label} clf] best ROC-AUC: {-study.best_value:.4f}')
    return {'objective': 'binary', 'metric': 'binary_logloss',
            'is_unbalance': True, **study.best_params}

In [ ]:
# ── Кэш артефактов ────────────────────────────────────────────────────

def _load_cached(name: str) -> dict | None:
    path = ARTIFACTS_DIR / f'{ARTIFACT_PREFIX}{name}.json'
    if not FORCE_RETUNE and path.exists():
        with path.open() as f:
            data = json.load(f)
        print(f'  [cache] loaded {path.name}')
        return data
    return None


def _save_meta(meta: dict, name: str) -> None:
    path = ARTIFACTS_DIR / f'{ARTIFACT_PREFIX}{name}.json'
    with path.open('w') as f:
        json.dump(meta, f, indent=2)
    print(f'  [cache] saved  {path.name}')

In [9]:
# ── Two-stage XGBoost ─────────────────────────────────────────────────

def twostage_xgb(X, y_raw, thr, label,
                 n_trials_reg=N_TRIALS_REG, n_trials_clf=N_TRIALS_CLF):
    y_label   = (y_raw > thr).astype(int)
    n_extreme = y_label.sum()
    print(f'\n--- {label} two-stage XGB (thr={thr}) ---')
    print(f'  extreme (>{thr}): {n_extreme} молекул ({n_extreme/len(y_raw):.1%})')

    cached = _load_cached(f'{label.lower()}_params')
    if cached:
        params_normal  = cached['params_normal']
        params_extreme = cached['params_extreme']
        params_clf     = cached['params_clf']
    else:
        print(f'  Tuning normal XGB {label}...')
        params_normal  = tune_xgb_reg(X, y_raw, n_trials_reg, f'{label} normal')
        print(f'  Tuning extreme XGB {label}...')
        params_extreme = tune_xgb_reg(X, y_raw, n_trials_reg, f'{label} extreme')
        print(f'  Tuning classifier {label}...')
        params_clf     = tune_xgb_clf(X, y_label, n_trials_clf, label)

    weights_sqrt = np.sqrt(y_raw) / np.sqrt(y_raw).mean()

    oof_normal,  test_normal  = xgb_oof(X, y_raw, params_normal)
    oof_extreme, test_extreme = xgb_oof_weighted(X, y_raw, params_extreme, weights_sqrt)
    oof_prob,    test_prob    = xgb_clf_oof(X, y_label, params_clf)

    print(f'  Normal OOF RMSE:    {rmse(y_raw, oof_normal):.4f}')
    print(f'  Extreme max pred:   {oof_extreme.max():.1f}  (true max: {y_raw.max():.1f})')
    print(f'  Classifier ROC-AUC: {roc_auc_score(y_label, oof_prob):.4f}')

    oof_ts  = (1 - oof_prob)  * oof_normal  + oof_prob  * oof_extreme
    test_ts = (1 - test_prob) * test_normal + test_prob * test_extreme
    print(f'  Two-stage OOF RMSE: {rmse(y_raw, oof_ts):.4f}')

    meta = {
        'params_normal':  params_normal,
        'params_extreme': params_extreme,
        'params_clf':     params_clf,
        'threshold':      thr,
        'rmse_normal':    float(rmse(y_raw, oof_normal)),
        'rmse_twostage':  float(rmse(y_raw, oof_ts)),
        'clf_auc':        float(roc_auc_score(y_label, oof_prob)),
    }
    if not cached:
        _save_meta(meta, f'{label.lower()}_params')
    return oof_ts, test_ts, meta

In [10]:
# ── Two-stage LightGBM (SI) ───────────────────────────────────────────

def twostage_lgb_si(X, y_raw, thr,
                    n_trials_reg=N_TRIALS_REG, n_trials_clf=N_TRIALS_CLF):
    """Two-stage LGB для SI.
    Extreme регрессор обучается на полном train: 16 молекул с SI>200 —
    OOF по ним слишком нестабилен для CV.
    """
    y_label   = (y_raw > thr).astype(int)
    n_extreme = y_label.sum()
    print(f'\n--- SI two-stage LGB (thr={thr}) ---')
    print(f'  extreme (>{thr}): {n_extreme} молекул ({n_extreme/len(y_raw):.1%})')

    cached = _load_cached('si_params')
    if cached:
        params_normal  = cached['params_normal']
        params_extreme = cached['params_extreme']
        params_clf     = cached['params_clf']
    else:
        print('  Tuning normal LGB SI...')
        params_normal  = tune_lgb_reg(X, y_raw, n_trials_reg, 'SI normal')
        print('  Tuning extreme LGB SI...')
        params_extreme = tune_lgb_reg(X, y_raw, n_trials_reg, 'SI extreme')
        print('  Tuning classifier SI...')
        params_clf     = tune_lgb_clf(X, y_label, n_trials_clf, 'SI')

    # Normal — OOF
    oof_normal, test_normal = lgb_oof(X, y_raw, params_normal)
    print(f'  Normal OOF RMSE: {rmse(y_raw, oof_normal):.4f}')

    # Extreme — full train (слишком мало extreme молекул для честного OOF)
    y_log        = log1p_target(y_raw)
    weights_sqrt = np.sqrt(y_raw) / np.sqrt(y_raw).mean()
    ext_model = lgb.train(
        {**params_extreme, 'seed': SEED, 'verbose': -1},
        lgb.Dataset(X, y_log, weight=weights_sqrt),
        num_boost_round=1000,
    )
    oof_extreme  = np.clip(expm1_pred(ext_model.predict(X)),      0, None)
    test_extreme = np.clip(expm1_pred(ext_model.predict(X_test)), 0, None)
    print(f'  Extreme max pred: {oof_extreme.max():.1f}  (true max: {y_raw.max():.1f})')

    # Classifier — OOF
    oof_prob, test_prob = lgb_clf_oof(X, y_label, params_clf)
    print(f'  Classifier ROC-AUC: {roc_auc_score(y_label, oof_prob):.4f}')

    oof_ts  = (1 - oof_prob)  * oof_normal  + oof_prob  * oof_extreme
    test_ts = (1 - test_prob) * test_normal + test_prob * test_extreme
    print(f'  Two-stage OOF RMSE: {rmse(y_raw, oof_ts):.4f}')

    meta = {
        'params_normal':  params_normal,
        'params_extreme': params_extreme,
        'params_clf':     params_clf,
        'threshold':      thr,
        'rmse_normal':    float(rmse(y_raw, oof_normal)),
        'rmse_twostage':  float(rmse(y_raw, oof_ts)),
        'clf_auc':        float(roc_auc_score(y_label, oof_prob)),
    }
    if not cached:
        _save_meta(meta, 'si_params')
    return oof_ts, test_ts, meta

---
## IC50 — Two-stage XGBoost (порог 500)

102 молекулы с IC50 > 500 дают **91% MSE**.

In [11]:
oof_ic50, test_ic50, ic50_meta = twostage_xgb(
    X_train, y_ic50, thr=THR_IC50, label='IC50',
)
print(f'\nIC50 final OOF RMSE: {rmse(y_ic50, oof_ic50):.4f}')


--- IC50 two-stage XGB (thr=500) ---
  extreme (>500): 94 молекул (14.9%)
  Tuning normal XGB IC50...


  0%|          | 0/120 [00:00<?, ?it/s]

[IC50 normal] best OOF RMSE: 310.4444
  Tuning extreme XGB IC50...


  0%|          | 0/120 [00:00<?, ?it/s]

[IC50 extreme] best OOF RMSE: 310.4444
  Tuning classifier IC50...


  0%|          | 0/60 [00:00<?, ?it/s]

[IC50 clf] best ROC-AUC: 0.8011
  Normal OOF RMSE:    312.1524
  Extreme max pred:   2144.7  (true max: 4095.2)
  Classifier ROC-AUC: 0.8011
  Two-stage OOF RMSE: 323.3830
  [cache] saved  final_ic50_params.json

IC50 final OOF RMSE: 323.3830


---
## CC50 — Two-stage XGBoost (порог 1500)

104 молекулы с CC50 > 1500 дают ~60% MSE.

In [ ]:
oof_cc50, test_cc50, cc50_meta = twostage_xgb(
    X_train, y_cc50, thr=THR_CC50, label='CC50',
)
print(f'\nCC50 final OOF RMSE: {rmse(y_cc50, oof_cc50):.4f}')


--- CC50 two-stage XGB (thr=1500) ---
  extreme (>1500): 31 молекул (4.9%)
  Tuning normal XGB CC50...


  0%|          | 0/120 [00:00<?, ?it/s]

[CC50 normal] best OOF RMSE: 437.5498
  Tuning extreme XGB CC50...


  0%|          | 0/120 [00:00<?, ?it/s]

---
## SI — Two-stage LightGBM (порог 200)

16 молекул с SI > 200 дают **100% MSE**.
Extreme регрессор обучается на полном train.

In [ ]:
oof_si, test_si, si_meta = twostage_lgb_si(
    X_train, y_si, thr=THR_SI,
)
print(f'\nSI final OOF RMSE: {rmse(y_si, oof_si):.4f}')

---
## Итоговые метрики

In [ ]:
scores = competition_score(y_ic50, oof_ic50, y_cc50, oof_cc50, y_si, oof_si)

print('OOF scores:')
for k, v in scores.items():
    print(f'  {k:12s} = {v:.4f}')

print()

print(f'OOF={scores["score"]:.2f}')

## Сохранение

In [ ]:
with (ARTIFACTS_DIR / f'{ARTIFACT_PREFIX}scores.json').open('w') as f:
    json.dump({k: float(v) for k, v in scores.items()}, f, indent=2)

submission = pd.DataFrame({
    'index': test[INDEX_COL],
    'IC50':  test_ic50,
    'CC50':  test_cc50,
    'SI':    test_si,
})
sub_path = SUBMISSIONS_DIR / f'{ARTIFACT_PREFIX}submission.csv'
submission.to_csv(sub_path, index=False)

print(f'Submission: {sub_path}')
print(f'IC50: min={test_ic50.min():.3f}  max={test_ic50.max():.1f}  mean={test_ic50.mean():.1f}')
print(f'CC50: min={test_cc50.min():.3f}  max={test_cc50.max():.1f}  mean={test_cc50.mean():.1f}')
print(f'SI:   min={test_si.min():.3f}   max={test_si.max():.1f}   mean={test_si.mean():.1f}')